# **The Chat Format**

In this notebook, you will explore how you can utilize the chat format to have extended conversations with chatbots personalized or specialized for specific tasks or behaviors.

## Setup

In [27]:
%pip install openai python-dotenv panel


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [59]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [69]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

def get_completion(prompt, model="gpt-3.5-turbo", temperature=0): 
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, 
    )
    return response.choices[0].message.content


def get_completion_from_messages(messages, model="gpt-3.5-turbo", temperature=0): 
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, 
    )
    return response.choices[0].message.content

In [61]:
messages =  [  
{'role':'system', 'content':'You are an assistant that speaks like Shakespeare.'},    
{'role':'user', 'content':'tell me a joke'},   
{'role':'assistant', 'content':'Why did the chicken cross the road'},   
{'role':'user', 'content':'I don\'t know'}  ]

In [70]:
response = get_completion_from_messages(messages, temperature=1)
print(response)

To get to the other side, good sir! Ah, 'tis a classic jest that never fails to amuse the masses.


In [74]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Hi, my name is Isa'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Hello Isa! How can I assist you today?


In [75]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Yes,  can you remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

I'm sorry, but I don't have the ability to remember or store personal information, including your name. Can I assist you with anything else today?


In [77]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},
{'role':'user', 'content':'Hi, my name is Isa'},
{'role':'assistant', 'content': "Hi Isa! It's nice to meet you. \
Is there anything I can help you with today?"},
{'role':'user', 'content':'Yes, you can remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Your name is Isa.


# OrderBot
We can automate the collection of user prompts and assistant responses to build a  OrderBot. The OrderBot will take orders at a pizza restaurant. 

In [56]:
def collect_messages(_):
    prompt = inp.value_input
    inp.value = ''
    context.append({'role':'user', 'content':f"{prompt}"})
    response = get_completion_from_messages(context) 
    context.append({'role':'assistant', 'content':f"{response}"})
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600, styles={'background-color': '#F6F6F6'})))
 
    return pn.Column(*panels)


In [88]:
import panel as pn  # GUI
pn.extension()

panels = [] # collect display 

context = [ {'role':'system', 'content':"""
You are OrderBot, an automated service to collect orders for a pizza restaurant. \
You first greet the customer, then collects the order, \
and then asks if it's a pickup or delivery. \
You wait to collect the entire order, then summarize it and check for a final \
time if the customer wants to add anything else. \
If it's a delivery, you ask for an address. \
Finally you collect the payment.\
Make sure to clarify all options, extras and sizes to uniquely \
identify the item from the menu.\
You respond in a short, very conversational friendly style. \
The menu includes \
pepperoni pizza  12.95, 10.00, 7.00 \
cheese pizza   10.95, 9.25, 6.50 \
eggplant pizza   11.95, 9.75, 6.75 \
fries 4.50, 3.50 \
greek salad 7.25 \
Toppings: \
extra cheese 2.00, \
mushrooms 1.50 \
sausage 3.00 \
canadian bacon 3.50 \
AI sauce 1.50 \
peppers 1.00 \
Drinks: \
coke 3.00, 2.00, 1.00 \
sprite 3.00, 2.00, 1.00 \
bottled water 5.00 \
"""} ]  # accumulate messages


inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)

dashboard

BokehModel(combine_events=True, render_bundle={'docs_json': {'b5159aaa-f8ce-4965-8019-e048538c9c36': {'version…

In [79]:
messages =  context.copy()
messages.append(
{'role':'system', 'content':'create a json summary of the previous food order. Itemize the price for each item\
 The fields should be 1) pizza, include size 2) list of toppings 3) list of drinks, include size   4) list of sides include size  5)total price '},    
)
 #The fields should be 1) pizza, price 2) list of toppings 3) list of drinks, include size include price  4) list of sides include size include price, 5)total price '},    

response = get_completion_from_messages(messages, temperature=0)
print(response)

{
  "pizza": {
    "type": "pepperoni pizza",
    "size": "large"
  },
  "toppings": [
    "extra cheese",
    "mushrooms"
  ],
  "drinks": [
    {
      "type": "coke",
      "size": "medium"
    }
  ],
  "sides": [
    {
      "type": "fries",
      "size": "regular"
    }
  ],
  "total price": 23.45
}


## Try experimenting on your own!

You can modify the menu or instructions to create your own orderbot!

# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [ ]:
messages =  context.copy()
messages.append(
{'role':'system', 'content':'''You are a coffee shop ordering assistant. \
Rules: \
- Ask for size (small, medium, large). \
- Ask if customer wants milk alternatives. \
- Suggest a pastry. \
- Summarize the final order. \
Menu:
Coffee:
- Americano: $3
- Latte: $4
- Cappuccino: $4

Pastries:
- Croissant: $2
- Muffin: $3'},    
'''})

response = get_completion_from_messages(messages, temperature=0)
print(response)



Hello! What size coffee would you like today - small, medium, or large? Would you like any milk alternatives with that? Also, may I suggest a pastry to go with your coffee?


In [87]:
messages =  context.copy()
messages.append(
{'role':'system', 'content':'''You are a pizza restaurant order assistant.

Your job is to:
- Greet customers politely.
- Show the menu when requested.
- Take orders accurately.
- Confirm the order before checkout.
- Calculate the total price.
- Never invent menu items.

Menu:
Pizza:
- Margherita: $10
- Pepperoni: $12
- Vegetarian: $11

Drinks:
- Coke: $2
- Water: $1

Desserts:
- Ice Cream: $4    
'''})

response = get_completion_from_messages(messages, temperature=0)
print(response)

Hello! How can I assist you today?


In [89]:
messages =  context.copy()
messages.append(
{'role':'system', 'content':'''You are a healthy restaurant assistant.

Rules:
- Recommend healthy options.
- Ask about allergies.
- Suggest a drink with every meal.
- Only use items from the menu.

Menu:
Salads:
- Caesar Salad: $9
- Greek Salad: $10

Protein Bowls:
- Chicken Bowl: $13
- Tofu Bowl: $12

Drinks:
- Green Tea: $3
- Water: $1    
'''})

response = get_completion_from_messages(messages, temperature=0)
print(response)

Hello! Looking for a healthy option today? We have a delicious Greek Salad for $10 or a Tofu Bowl for $12. Would you like to add a drink to your order? We have Green Tea for $3 or Water for $1. Let me know if you have any allergies or dietary preferences!


In [101]:
messages =  context.copy()
messages.append(
{'role':'system', 'content':'''You are a restaurant assistant.
Help customers order food'''})

response = get_completion_from_messages(messages, temperature=0)
print(response)

Hello! What can I get for you today?


# OrderBot Prompt Engineering Experiment

## Introduction

The goal of this exercise was to create and test multiple OrderBot prompts and evaluate their performance. Three different versions were developed: a Pizza OrderBot, a Healthy Restaurant OrderBot, and a Coffee Shop OrderBot.

## Results

The Pizza OrderBot performed well when taking straightforward orders. Because the menu and prices were clearly defined, the bot accurately calculated totals and confirmed orders.

The Healthy Restaurant OrderBot provided a better customer experience by asking follow-up questions about allergies and dietary preferences. This made the interaction feel more personalized.

The Coffee Shop OrderBot produced the most realistic conversations. It consistently asked for drink sizes and milk preferences before finalizing the order. The order summaries were also clear and accurate.

## Issues Encountered

A simplified version of the prompt that contained only general instructions performed poorly. Without a detailed menu and explicit rules, the model occasionally invented menu items and prices. This demonstrated the importance of providing sufficient context and constraints.

## Lessons Learned

This exercise showed that prompt quality has a significant impact on model behavior. Clear instructions, explicit menus, and well-defined rules reduce hallucinations and improve consistency. I also learned that system prompts can be used to guide the model's personality and behavior, while detailed constraints help ensure reliable outputs. Overall, prompt engineering is essential for creating robust conversational applications.
